# Interactive Dashboard for A/B Testing in Data Science Admissions: Applicant Demographics, Power Analysis, and Chi-Square Testing—Built with Dash, MongoDB, and a Class-Based Architecture

1. Mengimpor pandas, plotly, CountryConverter, komponen Dash, MongoClient, GofChisquarePower, scipy, Table2x2, Experiment, dan reset. Host MongoDB diatur ke localhost.

In [2]:
import pandas as pd
import plotly.express as px
from country_converter import CountryConverter
from dash import Dash, html, dcc, Input, Output, State
from pymongo import MongoClient
from statsmodels.stats.power import GofChisquarePower
import scipy.stats

from pymongo import MongoClient
from statsmodels.stats.contingency_tables import Table2x2
from wqulibs.ab_test.experiment import Experiment
from wqulibs.database import reset

MONGODB_HOST = "localhost"

2. Kelas MongoRepository (lapisan akses data). Metodenya:
    - get_nationality_value_counts() mengelompokkan pendaftar per negara (countryISO2), menambahkan nama negara dan kode ISO3, serta opsi normalisasi menjadi proporsi.
    - get_ages() menghitung usia dengan $dateDiff antara birthday dan $$NOW.
    - get_ed_value_counts() menghitung jumlah per tingkat pendidikan, diurutkan dari SMA sampai Doktor.
    - get_no_quiz_per_day() menghitung pendaftar harian yang belum menyelesaikan kuis (30 hari).
    -  get_contingency_table() menjalankan eksperimen lalu membuat tabel kontingensi 2×2.

In [3]:
class MongoRepository:
    """Provides data access methods for the ds-applicants collection.

    This class encapsulates all MongoDB queries needed by the
    dashboard. It connects to a local MongoDB instance on
    construction and exposes methods that return clean pandas
    objects for demographics, quiz statistics, and experiment
    results.

    Attributes
    ----------
    collection : pymongo.collection.Collection
        Reference to the ``ds-applicants`` collection in the
        ``wqu-abtest`` database.
    """

    def __init__(
        self,
        client=MongoClient(host=MONGODB_HOST, port=27017),
        db="wqu-abtest",
        collection="ds-applicants"
    ):
        # store the resolved collection as an attribute
        self.client = client
        self.db = client[db]
        self.collection = client[db][collection]

    def get_nationality_value_counts(self, normalize=True):
        """Aggregate applicant counts by nationality.

        Runs a MongoDB aggregation pipeline that groups documents
        by the ``countryISO2`` field and counts each group. The
        results are enriched with ISO-3 codes and full country
        names via the ``country_converter`` library.

        Parameters
        ----------
        normalize : bool, optional
            If ``True`` (default), divide each count by the total
            so that values represent proportions. If ``False``,
            return raw counts.

        Returns
        -------
        pandas.DataFrame
            DataFrame with columns ``country_iso2``,
            ``country_iso3``, ``country_name``, and ``count``.
            Rows are sorted by ``count`` in descending order.
        """
        result = self.collection.aggregate(
            [{"$group": {"_id": "$countryISO2", "count": {"$count": {}}}}]
        )
        
        df_nationality = (
            pd.DataFrame(result)
            .rename({"_id": "country_iso2"}, axis=1) # rename "_id" to "country_iso2"
            .sort_values("count", ascending=False)
        )
        
        cc = CountryConverter()
        df_nationality["country_name"] = cc.convert(
            df_nationality["country_iso2"], # pass the column with ISO2 codes
            to="name_short" # convert to short names
        )
        df_nationality["country_iso3"] = cc.convert(
            df_nationality["country_iso2"],  # pass the column with ISO2 codes
            to="ISO3"  # convert to three-letter ISO codes
        )

        if normalize:
            # compute proportion: count divided by total
            df_nationality["count_pct"] = (
                df_nationality["count"] / df_nationality["count"].sum()
            )
            df_nationality = df_nationality.drop(columns=["count"])

        return df_nationality


    def get_ages(self):
        """Compute applicant ages from their birth dates.

        Uses a ``$dateDiff`` aggregation to calculate the
        difference in years between each applicant's
        ``birthday`` field and the current date (``$$NOW``).

        Returns
        -------
        pandas.Series
            A Series of integer ages, one per applicant, with
            ``NaN`` values dropped.
        """
        
        result = self.collection.aggregate(
            [
                {
                    "$project": {
                        "years": {
                            "$dateDiff": {
                                "startDate": "$birthday",
                                "endDate": "$$NOW",
                                "unit": "year",
                            }
                        }
                    }
                }
            ]
        )

        # convert result to DataFrame and extract the "years" column
        ages = pd.DataFrame(result)["years"].dropna()

        return ages


    

    def get_ed_value_counts(self, normalize=False):
        """Count applicants by highest degree earned.

        Queries the ``highestDegreeEarned`` field and returns
        value counts reindexed in ascending educational order.

        Parameters
        ----------
        normalize : bool, optional
            If ``True``, return proportions instead of raw
            counts. Default is ``False``.

        Returns
        -------
        pandas.Series
            Counts (or proportions) indexed by degree name,
            ordered from ``"High School or Baccalaureate"`` to
            ``"Doctorate (e.g. PhD)"``.
        """
        result = self.collection.aggregate(
            [{"$group": {"_id": "$highestDegreeEarned", "count": {"$count": {}}}}]
        )

        education = (
            pd.DataFrame(result)
            .rename({"_id": "highest_degree_earned"}, axis=1)
            .set_index("highest_degree_earned")
            .squeeze()
        )

        degrees = [
            "High School or Baccalaureate",
            "Some College (1-3 years)",
            "Bachelor's degree",
            "Master's degree",
            "Doctorate (e.g. PhD)"
        ]
        # Create a dict mapping degree name -> position
        mapping = {deg: i for i, deg in enumerate(degrees)}
        
        # Build a list of positions for each entry in education.index
        sort_order = [mapping[deg] for deg in education.index]
        
        # Sort education
        education = education.iloc[
            sorted(range(len(sort_order)), key=lambda i: sort_order[i])
        ]
        if normalize:
            # compute proportion: count divided by total
            education = education / education.sum()
        return education


    def get_no_quiz_per_day(self):
        """Count daily new accounts that never completed the quiz.

        Filters for documents where ``admissionsQuiz`` is
        ``"incomplete"``, groups them by day using ``$dateTrunc``
        on ``createdAt``, and returns the first 30 days sorted
        chronologically.

        Returns
        -------
        pandas.Series
            A 30-element Series indexed by date with daily
            counts as values.
        """
        result = self.collection.aggregate(
            [
                {"$match": {"admissionsQuiz": "incomplete"}},
                {
                    "$group": {
                        "_id": {
                            "$dateTrunc": {
                                "date": "$createdAt",
                                "unit": "day",
                            }
                        },
                        "count": {"$sum": 1},
                    }
                },
            ]
        )
        
        no_quiz = (
            pd.DataFrame(result)
            .rename({"_id": "date", "count": "new_users"}, axis=1)  # rename "_id"->"date", "count"->"new_users"
            .set_index("date")
            .sort_index()
            .squeeze()
        )

        return no_quiz


    def get_contingency_table(self, days=10):
        """Build a 2x2 contingency table from experiment data.

        Queries documents where ``inExperiment`` is ``True`` and
        ``group`` is either ``"no email (control)"`` or
        ``"email (treatment)"``. Groups by ``group`` and
        ``admissionsQuiz`` status, then pivots into a 2x2
        DataFrame suitable for a chi-square test.

        Returns
        -------
        pandas.DataFrame
            A pivot table with experiment groups as the index
            and quiz completion status as the columns, with
            counts as the values.
        """

        exp = Experiment(
            repo=self.client,
            db=self.db.name,
            collection=self.collection.name,
        )

        exp.reset_experiment()
        exp.run_experiment(days=days)
        result = self.collection.find({"inExperiment": True})
        df = pd.DataFrame(result)
    
        data = pd.crosstab(
            index=df["group"],
            columns=df["admissionsQuiz"]
        )
    
        return data


3. Uji MongoRepository: koleksi ds-applicants valid. Negara terbanyak adalah Nigeria (1.238), Pakistan (568), India (511), Kenya (304), dan Bangladesh (254).

In [4]:
from pymongo.collection import Collection

repo = MongoRepository() # instantiate MongoRepository
assert isinstance(repo.collection, Collection)
print("Collection:", repo.collection.name)

df_nat = repo.get_nationality_value_counts(normalize=False)
print("Columns:", sorted(df_nat.columns.tolist()))
df_nat.head()

Collection: ds-applicants
Columns: ['count', 'country_iso2', 'country_iso3', 'country_name']


,country_iso2,count,country_name,country_iso3
26,NG,1238,Nigeria,NGA
5,PK,568,Pakistan,PAK
128,IN,511,India,IND
84,KE,304,Kenya,KEN
54,BD,254,Bangladesh,BGD


4. Memanggil get_ages() (5.025 data usia) dan get_ed_value_counts(). Mayoritas pendaftar berpendidikan Sarjana (2.643), diikuti Master (862), SMA (832), Some College (612), dan Doktor (76).

In [5]:
repo = MongoRepository()

ages = repo.get_ages()
print("Ages type:", type(ages).__name__, "| length:", len(ages))

degrees = repo.get_ed_value_counts(normalize=False)
print("Education index:", degrees.index.tolist())
degrees

Ages type: Series | length: 5025
Education index: ['High School or Baccalaureate', 'Some College (1-3 years)', "Bachelor's degree", "Master's degree", 'Doctorate (e.g. PhD)']


highest_degree_earned
High School or Baccalaureate     832
Some College (1-3 years)         612
Bachelor's degree               2643
Master's degree                  862
Doctorate (e.g. PhD)              76
Name: count, dtype: int64

5. get_no_quiz_per_day() menghasilkan 30 hari data (1 Mei 2022 = 37, 2 Mei = 49, dan seterusnya).

In [6]:
repo = MongoRepository()
no_quiz = repo.get_no_quiz_per_day()
print("Shape:", no_quiz.shape)
no_quiz.head()

Shape: (30,)


date
2022-05-01    37
2022-05-02    49
2022-05-03    43
2022-05-04    48
2022-05-05    47
Name: new_users, dtype: int64

6. assert untuk semua metode MongoRepository. Semua lolos.

In [7]:
from pymongo.collection import Collection

repo = MongoRepository()

assert isinstance(repo.collection, Collection), (
    f"Expected Collection, got {type(repo.collection).__name__}"
)
assert repo.collection.name == "ds-applicants", (
    f"Expected collection 'ds-applicants', "
    f"got '{repo.collection.name}'"
)

df_nat = repo.get_nationality_value_counts(normalize=False)
assert isinstance(df_nat, pd.DataFrame), (
    f"get_nationality_value_counts should return DataFrame, "
    f"got {type(df_nat).__name__}"
)
nat_cols = sorted(df_nat.columns.tolist())
expected_cols = ["count", "country_iso2", "country_iso3", "country_name"]
assert nat_cols == expected_cols, (
    f"Expected columns {expected_cols}, got {nat_cols}"
)

ages = repo.get_ages()
assert isinstance(ages, pd.Series), (
    f"get_ages should return Series, got {type(ages).__name__}"
)

degrees = repo.get_ed_value_counts(normalize=False)
assert isinstance(degrees, pd.Series), (
    f"get_ed_value_counts should return Series, "
    f"got {type(degrees).__name__}"
)
expected_ed = [
    "High School or Baccalaureate",
    "Some College (1-3 years)",
    "Bachelor's degree",
    "Master's degree",
    "Doctorate (e.g. PhD)",
]
assert degrees.index.tolist() == expected_ed, (
    f"Education index mismatch.\n"
    f"Expected: {expected_ed}\n"
    f"Got: {degrees.index.tolist()}"
)

no_quiz = repo.get_no_quiz_per_day()
assert isinstance(no_quiz, pd.Series), (
    f"get_no_quiz_per_day should return Series, "
    f"got {type(no_quiz).__name__}"
)
assert no_quiz.shape == (30,), (
    f"Expected shape (30,), got {no_quiz.shape}"
)

print("All MongoRepository checks passed.")

All MongoRepository checks passed.


7. Kelas GraphBuilder (lapisan visualisasi):
	- build_nat_choropleth() membuat peta dunia sebaran kebangsaan (warna oranye).
	- build_age_hist() membuat histogram usia.
	- build_ed_bar() membuat bar chart horizontal tingkat pendidikan.
	- build_contingency_bar() membuat bar chart berkelompok hasil eksperimen.

In [8]:
class GraphBuilder:
    """Builds Plotly figures using data from MongoRepository.

    This class provides chart-building methods for the three
    demographic visualizations and the contingency-table bar
    chart used in the Results section of the dashboard.

    Attributes
    ----------
    repo : MongoRepository
        Repository instance used to fetch data for each chart.
    """

    def __init__(self):
        """Instantiate a ``MongoRepository`` and store it.

        The repository is created once and reused by every
        chart-building method in this class.
        """
        self.repo = MongoRepository()  # instantiate MongoRepository

    def build_nat_choropleth(self):
        """Build a choropleth map of applicant nationalities.

        Fetches normalized nationality counts from the
        repository and renders them on a world map using
        ``plotly.express.choropleth``.

        Returns
        -------
        plotly.graph_objects.Figure
            A Plotly choropleth figure colored by applicant
            proportion per country.
        """
        fig = px.choropleth(
            data_frame=self.repo.get_nationality_value_counts(normalize=False),
            locations="country_iso3", # column with ISO3 codes
            color="count",
            projection="natural earth",
            color_continuous_scale=px.colors.sequential.Oranges, # check colors from px.colors.sequential
            title="DS Applicants: Nationalities"
        )
        return fig


    def build_age_hist(self):
        """Build a histogram of applicant ages.

        Fetches the age Series from the repository and
        renders a 30-bin histogram using
        ``plotly.express.histogram``.

        Returns
        -------
        plotly.graph_objects.Figure
            A Plotly histogram figure showing the age
            distribution.
        """
        fig = px.histogram(
            x=self.repo.get_ages(),
            nbins=20,
            title="Distribution of DS Applicant Ages",
        )
        fig.update_layout(
            xaxis_title="Age",
            yaxis_title="Frequency [count]"
        )
        return fig
        

    def build_ed_bar(self):
        """Build a bar chart of education levels.

        Fetches raw education counts from the repository and
        renders a vertical bar chart using
        ``plotly.express.bar``.

        Returns
        -------
        plotly.graph_objects.Figure
            A Plotly bar chart figure with education levels on
            the x-axis and counts on the y-axis.
        """
        fig = px.bar(
            x=self.repo.get_ed_value_counts().values,
            y=self.repo.get_ed_value_counts().index,
            orientation="h",
            title="DS Applicant Education Levels"
        )
        
        fig.update_layout(
            xaxis_title="Frequency [count]",
            yaxis_title="Highest Degree Earned"
        )
        return fig

    def build_contingency_bar(self):
        """Build a grouped bar chart from the contingency table.

        Fetches the 2x2 contingency table from the repository
        and renders a side-by-side (grouped) bar chart using
        ``plotly.express.bar``. This method should only be
        called after experiment data has been generated.

        Returns
        -------
        plotly.graph_objects.Figure
            A grouped bar chart showing quiz completion counts
            for each experiment group.
        """
        fig = px.bar(
            data_frame=self.repo.get_contingency_table(),
            barmode="group",
            title="Admissions Quiz Completion by Group",
        )
        fig.update_layout(
            xaxis_title="Group",
            yaxis_title="Frequency [count]",
            legend={"title": "Admissions Quiz"},
        )
        return fig

8. Menguji pembuatan peta, histogram usia, dan bar chart pendidikan. Masing-masing memastikan hasilnya berupa Figure.

In [9]:
from plotly.graph_objects import Figure

gb = GraphBuilder() # instantiate GraphBuilder
fig_choro = gb.build_nat_choropleth() # build the choropleth
assert isinstance(fig_choro, Figure)
fig_choro.show()

In [10]:
fig_age = gb.build_age_hist() # build the age histogram
assert isinstance(fig_age, Figure)
fig_age.show()

In [11]:
fig_ed = gb.build_ed_bar() # build the education bar chart
assert isinstance(fig_ed, Figure)
fig_ed.show()

9. assert gabungan untuk 3 grafik demografi. Lolos.

In [12]:
from plotly.graph_objects import Figure

gb = GraphBuilder()

for name in ["build_nat_choropleth", "build_age_hist", "build_ed_bar"]:
    fig = getattr(gb, name)()
    assert isinstance(fig, Figure), (
        f"{name} should return Figure, got {type(fig).__name__}"
    )

print("All GraphBuilder checks passed.")

All GraphBuilder checks passed.


10. Kelas StatsBuilder (lapisan statistik):
    - calculate_n_obs() menghitung total sampel yang dibutuhkan (power 0,8, alpha 0,05).
	- calculate_cdf_pct() menghitung peluang (%) mencapai jumlah sampel tersebut dalam sekian hari dengan CLT.
	- run_experiment() menjalankan simulasi eksperimen.
	- run_chi_square() menjalankan uji chi-square pada hasil eksperimen.

In [14]:
import math
import numpy as np
from wqulibs.ab_test.experiment import Experiment
from scipy import stats as st
from statsmodels.stats.contingency_tables import Table

class StatsBuilder:
    """Computes statistics for experiment monitoring.

    Provides methods for power analysis, experiment-duration
    estimation, experiment execution, and hypothesis testing.
    All data access is delegated to a ``MongoRepository``
    instance created on construction.

    Attributes
    ----------
    repo : MongoRepository
        Repository instance used to fetch experiment data.
    """

    def __init__(self):
        """Instantiate a ``MongoRepository`` and store it.

        The repository is created once and shared across all
        statistical methods in this class.
        """
        self.repo = MongoRepository()  # instantiate MongoRepository

    def calculate_n_obs(self, effect_size):
        """Calculate the required number of observations per group.

        Uses the ``GofChisquarePower`` class from statsmodels
        to solve for the minimum sample size needed to detect
        the given ``effect_size`` with alpha = 0.05 and
        power = 0.8 in a two-bin chi-square test. The raw
        per-bin result is rounded up with ``math.ceil`` and
        then doubled to give the total observations needed
        across both groups.

        Parameters
        ----------
        effect_size : float
            The minimum detectable effect size (Cohen's *w*).
            Common benchmarks: 0.2 (small), 0.5 (medium),
            0.8 (large).

        Returns
        -------
        int
            Total number of observations required (both groups
            combined).
        """
        chi_square_power = GofChisquarePower()  # instantiate GofChisquarePower
        group_size = math.ceil(
            chi_square_power.solve_power(
                effect_size=effect_size, alpha=0.05, power=0.8
            )
        )
        return group_size * 2

    def calculate_cdf_pct(self, n_obs, days):
        """Estimate the probability of collecting enough data.

        Retrieves daily no-quiz applicant counts from the
        repository, computes the mean and standard deviation,
        and uses the central limit theorem to model the total
        arrivals over the specified number of ``days``. Returns
        the probability (as a percentage) that the total will
        reach or exceed ``n_obs``.

        Parameters
        ----------
        n_obs : int
            Target number of observations to collect.
        days : int
            Number of days the experiment would run.

        Returns
        -------
        float
            Probability (0–100) of reaching ``n_obs``
            observations within the given ``days``.
        """
        no_quiz = self.repo.get_no_quiz_per_day()
        daily_mean, daily_std = no_quiz.mean(), no_quiz.std()
        
        # Sum distribution over `days` days
        sum_mean = daily_mean * days
        sum_std = daily_std * np.sqrt(days)
        
        # P(Total >= target) = 1 - CDF(target)
        p_reach = 1 - scipy.stats.norm.cdf(
            n_obs, loc=sum_mean, scale=sum_std
        )
        return p_reach * 100

    def run_experiment(self, days):
        """Run a simulated A/B experiment.

        Creates an ``Experiment`` instance connected to
        ``self.repo`` and runs it for the specified number
        of days. The generated student records are inserted
        into the MongoDB collection.

        Parameters
        ----------
        days : int
            Number of simulated days to run the experiment.
        """
        client=self.repo.client
        db=self.repo.db.name
        collection=self.repo.collection.name

        exp = Experiment(
            repo=client,
            db=db,
            collection=collection,
        )

        exp.reset_experiment()
        exp.run_experiment(days=days)

    def run_chi_square(self):
        """Perform a chi-square test on the experiment results.

        Fetches the 2x2 contingency table from the repository,
        wraps it in a ``statsmodels.stats.contingency_tables.Table``,
        and runs ``test_nominal_association`` to test whether
        the experiment group and quiz completion are independent.

        Returns
        -------
        statsmodels.stats.contingency_tables._Bunch
            Test result containing the chi-square statistic,
            p-value, and degrees of freedom.
        """
        result = self.repo.collection.find({"inExperiment": True})
        df = pd.DataFrame(result)
        data = pd.crosstab(
            index= df["group"],  # the group column
            columns=df["admissionsQuiz"],  # the quiz completion column
            normalize=False,
        )
        
        contingency_table = Table2x2(data.values)
        
        chi_square_test = (
            contingency_table.test_nominal_association()
        )

        return chi_square_test


11. calculate_n_obs(0.2) = 394 observasi total. (Pesan cetaknya menyebut “per group”, padahal 394 adalah total.)

In [15]:
sb = StatsBuilder() # instantiate StatsBuilder
n_obs = sb.calculate_n_obs(effect_size=0.2)
print("Required observations per group:", n_obs)

Required observations per group: 394


12. calculate_cdf_pct(394, 12 hari) = 99,99999972%, hampir pasti tercapai.

In [16]:
pct = sb.calculate_cdf_pct(n_obs=394, days=12)
print(f"Probability of enough observations: {pct}%")

Probability of enough observations: 99.99999972153115%


13. Menguji run_experiment(days=1): 110 dokumen ditambahkan, lalu database di-reset (110 dihapus).

In [17]:
mr = MongoRepository()

# exp = Experiment(repo=mr)
exp = Experiment(
    repo=mr.client,
    db=mr.db.name,
    collection=mr.collection.name
)

exp.reset_experiment()

docs_before = mr.collection.count_documents({})
sb.run_experiment(days=1)
docs_after = mr.collection.count_documents({})
print("Documents added:", docs_after - docs_before)

exp.reset_experiment()

Documents added: 110


{'acknowledged': True, 'deleted_count': 110}

14. Menjalankan eksperimen 1 hari dan uji chi-square. Hasil p-value = 0,576 (tidak signifikan).

In [18]:
sb.run_experiment(days=1)
result = sb.run_chi_square() # run chi-square test
print("p-value:", result.pvalue)
exp.reset_experiment()

p-value: 0.5758111991901154


{'acknowledged': True, 'deleted_count': 189}

15. Membuat bar chart kontingensi lewat GraphBuilder.

In [19]:
gb = GraphBuilder()
fig_ct = gb.build_contingency_bar() # build the contingency bar chart
assert isinstance(fig_ct, Figure)
fig_ct.show()

16. assert untuk StatsBuilder (n_obs = 394, pct > 95, dokumen bertambah, hasil bertipe _Bunch). Lolos.

In [20]:
from wqulibs.ab_test.experiment import Experiment
from statsmodels.stats.contingency_tables import _Bunch

sb = StatsBuilder()

# calculate_n_obs
n_obs = sb.calculate_n_obs(effect_size=0.2)
assert isinstance(n_obs, int), (
    f"calculate_n_obs should return int, got {type(n_obs).__name__}"
)
assert n_obs == 394, (
    f"Expected 394 for effect_size=0.2, got {n_obs}"
)

# calculate_cdf_pct
pct = sb.calculate_cdf_pct(n_obs=394, days=12)
assert isinstance(pct, float), (
    f"calculate_cdf_pct should return float, "
    f"got {type(pct).__name__}"
)
assert 95 < pct <= 100, (
    f"Expected pct in (95, 100], got {pct}"
)

# run_experiment
mr = MongoRepository()
# exp = Experiment(repo=mr)
exp = Experiment(
    repo=mr.client,
    db=mr.db.name,
    collection=mr.collection.name
)
exp.reset_experiment()
docs_before = mr.collection.count_documents({})
sb.run_experiment(days=1)
docs_after = mr.collection.count_documents({})
assert docs_after > docs_before, (
    "run_experiment should add documents to the collection"
)

# run_chi_square
result = sb.run_chi_square()
assert isinstance(result, _Bunch), (
    f"run_chi_square should return _Bunch, "
    f"got {type(result).__name__}"
)

exp.reset_experiment()
print("All StatsBuilder checks passed.")

All StatsBuilder checks passed.


17. Membuat aplikasi Dash dengan tiga bagian: Applicant Demographics (dropdown Nationality/Age/Education), Experiment (slider effect size 0,1–1,0 dan slider durasi 1–30 hari), dan Results (tombol Start Experiment). Juga mendefinisikan 4 callback:
	- display_demo_graph menampilkan grafik sesuai pilihan dropdown.
	- display_group_size menampilkan jumlah observasi yang dibutuhkan.
	- display_cdf_pct menampilkan peluang mencapai jumlah sampel dalam N hari.
	- display_results menjalankan eksperimen saat tombol diklik, lalu menampilkan grafik dan hasil chi-square (statistik, p-value, df).

In [22]:
from dash import Dash, html, dcc, Input, Output, State

# Create the Dash application
app = Dash(__name__)  # instantiate Dash(__name__)

# Define the layout
app.layout = html.Div([
    # --- Applicant Demographics ---
    html.H1("Applicant Demographics"),
    # Add Dropdown (id="demo-plots-dropdown") and
    # Div (id="demo-plots-display") here
    dcc.Dropdown(
        id="demo-plots-dropdown",
        options=["Nationality", "Age", "Education"],
        value="Nationality",
    ),
    html.Div(id="demo-plots-display"),


    # --- Experiment ---
    html.H1("Experiment"),
    # Add effect-size Slider (id="effect-size-slider"),
    # Div (id="effect-size-display"),
    # days Slider (id="experiment-days-slider"),
    # Div (id="experiment-days-display") here
    # Effect size
    html.H2("Effect Size"),
    dcc.Slider(
        min=0.1,
        max=1.0,
        step=0.1,
        value=0.2,
        id="effect-size-slider"
    ),
    html.Div(id="effect-size-display"),
    
    # Experiment duration
    html.H2("Experiment Duration (days)"),
    dcc.Slider(
        min=1,
        max=30,
        step=1,
        value=10,
        id="experiment-days-slider"
    ),
    html.Div(id="experiment-days-display"),


    # --- Results ---
    html.H1("Results"),
    # Add Button (id="start-experiment-button") and
    # Div (id="results-display") here
    html.Button(
        "Start Experiment",
        id="start-experiment-button"
    ),
    html.Div(id="results-display"),
])

# Callback: demographic chart
@app.callback(
    Output("demo-plots-display", "children"),
    Input("demo-plots-dropdown", "value"),
)
def display_demo_graph(selection):
    """Return the selected demographic chart.

    Uses a ``GraphBuilder`` to construct the chart that
    matches the user's drop-down selection.

    Parameters
    ----------
    selection : str
        One of ``"Nationality"``, ``"Age"``, or
        ``"Education"``.

    Returns
    -------
    dash.dcc.Graph
        A Dash ``Graph`` component wrapping the Plotly figure.
    """
    gb = GraphBuilder() # instantiate GraphBuilder
    if selection == "Nationality":
        fig = gb.build_nat_choropleth() # build the choropleth
    elif selection == "Age":
        fig = gb.build_age_hist()
    elif selection == "Education":
        fig= gb.build_ed_bar()
    else:
        return html.P("Please select a visualization.")

    return dcc.Graph(figure=fig)



# Callback: group size from effect-size slider
@app.callback(
    Output("effect-size-display", "children"),
    Input("effect-size-slider", "value"),
)
def display_group_size(effect_size):
    """Display the required sample size for the given effect size.

    Instantiates a ``StatsBuilder``, calls ``calculate_n_obs``,
    and returns a paragraph showing the result.

    Parameters
    ----------
    effect_size : float
        Value from the effect-size slider (0.1–1.0).

    Returns
    -------
    dash.html.P
        A paragraph element stating the required observations.
    """
    sb = StatsBuilder()

    group_size = sb.calculate_n_obs(effect_size)

    return html.P(
        f"Required observations: {group_size}"
    )

# Callback: CDF percentage from both sliders
@app.callback(
    Output("experiment-days-display", "children"),
    Input("experiment-days-slider", "value"),
    Input("effect-size-slider", "value"),
)
def display_cdf_pct(days, effect_size):
    """Display the probability of collecting enough observations.

    Instantiates a ``StatsBuilder``, computes the required
    sample size and the CDF percentage for the given number
    of days, and returns a paragraph with the result.

    Parameters
    ----------
    days : int
        Value from the experiment-days slider (1–30).
    effect_size : float
        Value from the effect-size slider (0.1–1.0).

    Returns
    -------
    dash.html.P
        A paragraph element stating the probability percentage.
    """
    sb = StatsBuilder()

    n_obs = sb.calculate_n_obs(effect_size)

    cdf_pct = sb.calculate_cdf_pct(
        n_obs=n_obs,
        days=days
    )

    return html.P(
        f"Probability of reaching {n_obs} observations "
        f"in {days} days: {cdf_pct:.2f}%"
    )

# Callback: run experiment and show results
@app.callback(
    Output("results-display", "children"),
    Input("start-experiment-button", "n_clicks"),
    State("experiment-days-slider", "value"),
)
def display_results(n_clicks, days):
    """Run the experiment and display results.

    Triggered when the user clicks the "Run Experiment"
    button. Runs the A/B experiment for the specified
    number of days, builds the contingency-table bar chart,
    performs the chi-square test, and returns a ``Div``
    containing the chart and test summary.

    Parameters
    ----------
    n_clicks : int or None
        Number of times the button has been clicked.
        ``None`` on initial page load.
    days : int
        Value from the experiment-days slider (via ``State``).

    Returns
    -------
    dash.html.Div or dash.html.P
        A ``Div`` containing the bar chart and test results,
        or a placeholder ``P`` if the button has not been
        clicked yet.
    """
    if not n_clicks:
        return html.P("Click Start Experiment to begin.")

    sb = StatsBuilder()
    gb = GraphBuilder()

    # Run experiment
    sb.run_experiment(days=days)

    # Build contingency chart
    fig = gb.build_contingency_bar()

    # Perform chi-square test
    result = sb.run_chi_square()

    # Display results
    return html.Div([
        dcc.Graph(figure=fig),

        html.H3("Chi-Square Test Results"),

        html.P(f"Chi-square statistic: {result.statistic:.4f}"),

        html.P(f"P-value: {result.pvalue:.4f}"),

        html.P(f"Degrees of freedom: {result.df}"),
    ])

18. assert bahwa semua ID komponen ada di layout.

In [23]:
# Verify app exists and has key components
assert hasattr(app, "layout"), "app must have a layout attribute"

# Flatten layout to string representation for ID checks
layout_str = str(app.layout)

expected_ids = [
    "demo-plots-dropdown",
    "demo-plots-display",
    "effect-size-slider",
    "effect-size-display",
    "experiment-days-slider",
    "experiment-days-display",
    "start-experiment-button",
    "results-display",
]

for comp_id in expected_ids:
    assert comp_id in layout_str, (
        f"Component with id '{comp_id}' not found in layout"
    )

print("All layout checks passed.")

All layout checks passed.


19. app.run(debug=True) menjalankan dasbor (tampil dalam IFrame).

In [24]:
# # app.run(host="127.0.0.1", port=9001, jupyter_mode="inline", debug=True)

# app.run(host="127.0.0.1", port=9000, debug=False)

# import webbrowser

# webbrowser.open("http://127.0.0.1:9000")


app.run(debug=True)

20. Pengingat untuk menguji dasbor secara visual.

In [25]:
print(
    "Visual checkpoint: interact with the dashboard above\n"
    "and verify all four behaviors described in the list."
)

Visual checkpoint: interact with the dashboard above
and verify all four behaviors described in the list.


21. Kode (dikomentari) untuk me-reset database eksperimen.

In [ ]:
# kode untuk reset database
# exp = Experiment(
#     repo=MongoClient(host="localhost", port=27017),
#     db="wqu-abtest",
#     collection="ds-applicants"
# )
# exp.reset_experiment()

Kesimpulan Singkat

- Tujuan: 
    - Membangun dasbor web interaktif yang menggabungkan seluruh alur proyek sebelumnya: demografi pendaftar, perencanaan eksperimen (power analysis), pelaksanaan simulasi A/B, dan pengujian hasil dengan chi-square.
- Arsitektur tiga lapis:
    - MongoRepository mengurus data (query MongoDB).
    - GraphBuilder mengurus visualisasi.
    - StatsBuilder mengurus perhitungan statistik.
    - Aplikasi Dash hanya memanggil ketiganya lewat callback, sehingga kode modular dan mudah diuji.
- Temuan demografi: Dari 5.025 pendaftar, mayoritas dari Nigeria, Pakistan, India, Kenya, dan Bangladesh, dengan pendidikan terbanyak Sarjana (2.643, ±53%).
- Perencanaan eksperimen: Untuk effect size kecil (0,2) diperlukan 394 observasi total. Dengan rata-rata harian pendaftar yang belum menyelesaikan kuis, eksperimen 12 hari hampir pasti (≈100%) mengumpulkan sampel sebanyak itu.
- Hasil uji contoh: Pada percobaan 1 hari, p-value = 0,576 sehingga tidak ada bukti bahwa email pengingat meningkatkan penyelesaian kuis. Hasil ini konsisten dengan analisis sebelumnya (p = 0,427).

Catatan kritis:
- Pesan cetak sel 13 (“per group”) salah. Nilai 394 adalah total untuk kedua grup (197 per grup).
- MongoClient(...) sebagai nilai default parameter di MongoRepository.__init__ dibuat saat kelas didefinisikan, bukan saat objek dibuat. Lebih baik memakai None lalu membuat klien di dalam __init__.
- GraphBuilder dan StatsBuilder membuat MongoRepository baru di setiap callback, dan get_ed_value_counts() dipanggil dua kali di build_ed_bar(). Ini boros query.
- get_contingency_table() dan display_results memanggil run_experiment yang me-reset dan menyuntikkan data baru. Akibatnya, display_results menjalankan eksperimen dua kali per klik (sekali oleh StatsBuilder.run_experiment dan sekali lagi lewat build_contingency_bar → get_contingency_table), sehingga grafik dan hasil chi-square bisa berasal dari data yang berbeda.
Docstring build_age_hist menyebut 30 bin, tetapi kodenya memakai nbins=20. Docstring build_ed_bar menyebut “vertical”, padahal grafiknya horizontal.
- Di run_chi_square, kelas yang diimpor (Table) berbeda dari yang dipakai (Table2x2). Tidak error, tetapi bisa membingungkan.
app.run(debug=True) tidak dianjurkan untuk produksi.